# Lindenberg radiation closure

*Kerstin Ebell (kerstin.ebell@uni-koeln.de), Andreas Walbröl (a.walbroel@uni-koeln.de)*

Introduction?

Goals?


### Preparation

1. Activate the correct python environment (handled with conda): conda activate ...
2. Data: 


### Modification options:

When modifying the meteorological data, remember to stick to SI-units (e.g., Pa instead of hPa, K instead of °C, ...).
1. Already implemented: 
    - replace model temperature near the surface by observed 2 m temperatures
    - scale specific humidity profile by measured HATPRO integrated water vapour (IWV)

Below, define paths where input data is located, where output is saved to and where to save plots.

### Paths

In [ ]:
# _paths.py

import os

os.environ['PATH_DATA_BASE'] = "/data/lindenberg_radiation/"
os.environ['PATH_PLOTS_BASE'] = os.path.expanduser("~")

path_radiation_sim = os.path.join(os.environ['PATH_DATA_BASE'],
                                  "radiation_simulations/")
path_tcars_data = os.path.join(os.environ['PATH_DATA_BASE'],
                               "tcars_data/")
path_cloudnet_data = os.path.join(os.environ['PATH_DATA_BASE'],
                                 "cloudnet/")
path_radiation_obs = os.path.join(os.environ['PATH_DATA_BASE'],
                                  "radiation/")
path_meteo_obs = os.path.join(os.environ['PATH_DATA_BASE'],
                              "meteo/")
path_hatpro_data = os.path.join(os.environ['PATH_DATA_BASE'],
                                "hatpro/")


Below, necessary constants and functions for the simulations are defined and loaded. These cells should not be modified unless you really want to.

In [ ]:
import sys
from copy import deepcopy
from collections import OrderedDict as odict

import numpy as np
import xarray as xr
import csv
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from cmcrameri import cm
from skyfield import api
from skyfield.api import Loader
import pyrrtmg as rrtmg

In [ ]:
# constants.py

R_d = 287.0597  # gas constant of dry air, in J kg-1 K-1
R_v = 461.5     # gas constant of water vapour, in J kg-1 K-1
R_ = 8.314462618    # universal gas constant, in J mol-1 K-1, https://physics.nist.gov/cgi-bin/cuu/Value?r
M_dv = R_d / R_v # molar mass ratio , in ()

m_mol_air = 0.0289647       # molar mass of dry air, in kg mol-1, https://www.engineeringtoolbox.com/molecular-mass-air-d_679.html
mw_h2o = 0.01802  # h2o molar mass in kg mol-1
mw_o3 = 0.0479982 # ozone molar mass in kg mol-1
mw_co2 = 0.0440   # co2 molar mass in kg mol-1
mw_n2o = 0.01802  # n2o molar mass in kg mol-1
mw_co = 0.02801   # co molar mass in kg mol-1
mw_ch4 = 0.01604  # ch4 molar mass in kg mol-1
mw_o2 = 0.015999  # o2 molar mass in kg mol-1

e_0 = 611       # saturation water vapour pressure at freezing point (273.15 K), in Pa
T0 = 273.15     # freezing temperature, in K
c_pd = 1005.7   # specific heat capacity of dry air at constant pressure, in J kg-1 K-1
c_vd = 719.0    # specific heat capacity of dry air at constant volume, in J kg-1 K-1
c_h2o = 4187.0  # specific heat capacity of water at 15 deg C; in J kg-1 K-1
L_v = 2.501e+06 # latent heat of vaporization, in J kg-1
sigma_sb = 5.670374419e-08  # Stefan-Boltzmann constant W m-2 K-4, 
                            # https://physics.nist.gov/cgi-bin/cuu/Value?sigma|search_for=stefan

g = 9.80665     # gravitation acceleration, in m s^-2 (from https://doi.org/10.6028/NIST.SP.330-2019 )
omega_earth = 2*np.pi / 86164.09    # earth's angular velocity: World Book Encyclopedia Vol 6. Illinois: World Book Inc.: 1984: 12.
R_e = 6371000   # volumetric radius of earth in m, https://nssdc.gsfc.nasa.gov/planetary/factsheet/earthfact.html

sfac = {'ppm': 1e-06, 'ppb': 1e-09, 'ppt': 1e-12}   # acronyms, scaling factor for ppm, ppb, ...

solar_constant = 1368.      # Solar constant, in W m-2 ; 
                            # some sources suggest 1367 W m-2 (https://doi.org/10.1016/B978-0-08-087872-0.00302-4)

In [ ]:
# data_tools.py

def running_mean_pdtime(x, N, t):
    
    """
    Running mean of a 1D array x with a window width of N seconds.

    Parameters:
    -----------
    x : array of floats
        1D data vector of which the running mean is to be taken.
    N : int
        Running mean window width in seconds.
    t : array of floats
        1D time vector (in numpy datetime64[ns]) required to
        compute the actual running mean window width.
    """

    # first, create xarray DataArray and convert it to pandas DataFrame:
    x_DA = xr.DataArray(x, dims=['time'], coords={'time': (['time'], t)})
    x_DF = x_DA.to_dataframe(name='x')

    # compute running mean (rolling mean): center=True is recommended to have a 5-min running
    # at 2020-01-01T14:00:00 from 2020-01-01T13:57:30 until 2020-01-01T14:02:30.
    x_rm = x_DF.rolling(f"{int(N)}S", center=True).mean().to_xarray().x

    return x_rm.values


def encode_time(
    DS: xr.Dataset, 
    time_var='time', 
    time_dim='time', 
    reference_period=np.datetime64("1970-01-01T00:00:00"),
    calendar="proleptic_gregorian"):
    
    """
    Encode the time dimension of a Dataset with respect to a given reference period.
    
    Parameters:
    -----------
    DS : xr.Dataset
        Dataset whose time dimension should be encoded.
    time_var : str
        Name of the time variable to be encoded
    time_dim : str
        Name of the time dimension of the variable to be encoded.
    reference_period : np.datetime64
        Reference period as np.datetime64 object, given in YYYY-MM-DDThh:mm:ss (ISO 8601).
    calendar : str
        String describing the calendar used. Numpy's datetime64 uses 'proleptic_gregorian'.
        See also https://cfconventions.org/cf-conventions/cf-conventions.html#calendar .
    """
    
    reference_period_str = str(reference_period).replace("T", " ")
    
    time_values = (DS[time_var].values - reference_period).astype('timedelta64[s]').astype(np.float64)
    if time_var == time_dim:
        DS[time_var] = time_values
    else:
        DS[time_var] = xr.DataArray(time_values, dims=time_dim)
    DS[time_var].attrs['units'] = f"seconds since {reference_period_str}"
    DS[time_var].encoding['units'] = f'seconds since {reference_period_str}'
    DS[time_var].encoding['dtype'] = 'double'
    
    if time_var == 'time': DS[time_var].attrs['standard_name'] = 'time'
    DS[time_var].attrs['calendar'] = calendar
    
    return DS


def write_basic_attributes(DS: xr.Dataset):
    
    DS.attrs['institution'] = "Institute for Geophysics and Meteorology, University of Cologne, Cologne, Germany"
    DS.attrs['contact'] = "Andreas Walbroel (a.walbroel@uni-koeln.de, https://orcid.org/0000-0003-2603-2724)"
    DS.attrs['author'] = "Andreas Walbroel"
    DS.attrs['licence'] = "CC BY-NC 4.0, https://creativecommons.org/licenses/by-nc/4.0/"
    
    return DS


def update_netCDF_file_history(
    DS: xr.Dataset, 
    script_name: str, 
    summary_str="", 
    history_attr='history',
    history_attr_exists=True):

    """
    Updates the history of an xarray Dataset that shall be saved to a netCDF file.
    
    Parameters:
    -----------
    DS : xr.Dataset
        Dataset to be saved and where the attribute is added.
    script_name : str
        Name of the script that was mainly used to update/modify DS.
    summary_str : str
        String that concisely describes the changes made to DS.
    history_attr : str
        Name of the attribute where the history of DS is described.
    history_attr_exists : bool
        Boolean indicating whether the hisotry attribute already exists. If False,
        it's created.
    """
    
    if not history_attr_exists: DS.attrs[history_attr] = ""

    attr_add = ""
    if history_attr_exists and (";" not in DS.attrs[history_attr][-2:]):
        attr_add = "; "
    DS.attrs[history_attr] += (f"{attr_add}{str(np.datetime64('now')).replace('T', ' ')}" +
                               f", {summary_str} with {script_name}; ")
    
    return DS


def convert_units(data: np.ndarray, unit_conv_list: list):
    
    """
    Convert some units: first (second) element of list: must be added to the data (the data 
    must be multiplied by) to get to the desired unit. The multiplication is performed after 
    adding the unit_conv_list[0] value.
    """
    
    return (data + unit_conv_list[0])*unit_conv_list[1]


def convert_units_back(data: np.ndarray, unit_conv_list: list):
    
    """
    Inverse of 'convert_units'. Undo conversion changes.
    """
    
    return (data / unit_conv_list[1]) - unit_conv_list[0]


def identify_files_daterange(path: str, daterange: np.ndarray, file_pattern: str, yyyymmdd_delim=""):
    
    """    
    Parameters:
    -----------
    path : str
        Full path where files containing the data are located.
    daterange : np.ndarray
        Array of np.datetime64 indicating the date range.
    file_pattern : str
        String indicating the file pattern of the data.
    yyyymmdd_delim : str
        Delimiter used between year, month and days in the date strings of the data files.
    """
    
    daterange = daterange.astype('datetime64[D]')

    files = list()
    for date in daterange:
        date_str = str(date).replace("-", yyyymmdd_delim)
        file = glob.glob(path + file_pattern.replace("__DATE_STRING__", date_str))
        if len(file) >= 1:
            files.extend(file)
    
    return files


def handle_daterange_or_date_start_end(
    daterange=None, 
    date0=None, 
    date1=None,
    default_daterange=np.array([np.datetime64("2000-01-01")])):
    
    if (daterange is None) and (date0 is None) and (date1 is None):
        daterange = default_daterange
    elif (daterange is None) and ((date0 is not None) and (date1 is not None)):
        daterange = np.arange(np.datetime64(date0), np.datetime64(date1) + np.timedelta64(1, "D"),
                              np.timedelta64(1, "D"))
    elif (daterange is None) and (date0 is not None) and (date1 is None):
        daterange = np.array([np.datetime64(date0)])
        
    return daterange


def get_files_daterange_filepattern(
    path_data: str,
    date0=None,
    date1=None,
    daterange=None,
    default_daterange=np.array([np.datetime64("2001-01-01")]),
    file_pattern="*__DATE_STRING__*.nc",
    yyyymmdd_delim=""):
    
    daterange = handle_daterange_or_date_start_end(daterange, 
                                                   date0,
                                                   date1,
                                                   default_daterange=default_daterange)
    
    files = identify_files_daterange(path=path_data,
                                     daterange=daterange,
                                     file_pattern=file_pattern,
                                     yyyymmdd_delim=yyyymmdd_delim)
    
    return files

In [ ]:
# met_tools.py

def compute_IWV_q(
    q,
    press,
    nan_threshold=0.0,
    scheme='balanced'):

    """
    Compute Integrated Water Vapour (also known as precipitable water content)
    out of specific humidity (in kg kg^-1), gravitational constant and air pressure (in Pa).
    The moisture data may contain certain number gaps (up to nan_threshold*n_levels) but
    the height variable must be free of gaps.

    Parameters:
    -----------
    q : array of floats
        One dimensional array of specific humidity in kg kg^-1.
    press : array of floats
        One dimensional array of pressure in Pa.
    nan_threshold : float, optional
        Threshold describing the fraction of nan values of the total height level
        number that is still permitted for computation.
    scheme : str, optional
        Chose the scheme 'balanced' or 'top_weighted'. They differ in the way the altitude
        levels are used to compute IWV. Recommendation and default: 'balanced'
    """

    IWV = np.nan

    # Check if the Pressure axis is sorted in descending order:
    if np.any(np.diff(press) > 0):
        print("Warning! Height axis must be in ascending order (pressure in descending) to compute the integrated" +
            " water vapour.")

        # if the pressure data is okay until 300 hPa, compute IWV nonetheless and truncate the
        # profile beyond:
        where_broken = np.where(np.diff(press) > 0)[0]      # when where_broken == 152, then press[153] - press[152] is broken
        if press[where_broken[0]] > 30000.0:    # then, sufficient altitude doesn't have valid data valid data, return IWV=nan
            return IWV

    # truncate data to non nan height or pressure levels:
    non_nan_idx = np.where(~np.isnan(press))[0]
    q = q[non_nan_idx[0]:non_nan_idx[-1]+1]
    press = press[non_nan_idx[0]:non_nan_idx[-1]+1]

    # check if height axis is free of gaps:
    if np.any(np.isnan(np.diff(press))): 
        print("Height axis contains gaps. Aborted IWV computation.")
        return IWV


    n_height = len(press)
    # Check if q has got any gaps:
    n_nans = np.count_nonzero(np.isnan(q))


    # If no nans exist, the computation is simpler. If some nans exist IWV will still be
    # computed but needs to look for the next non-nan value. If too many nans exist IWV
    # won't be computed.
    if scheme == 'balanced':
        if (n_nans == 0):

            IWV = 0.0
            for k in range(n_height):
                if k == 0:      # bottom of grid
                    dp = 0.5*(press[k+1] - press[k])        # just a half of a level difference

                elif k == n_height-1:   # top of grid
                    dp = 0.5*(press[k] - press[k-1])        # the other half level difference

                else:           # mid of grid
                    dp = 0.5*(press[k+1] - press[k-1])

                IWV = IWV - q[k]*dp

        elif n_nans / n_height < nan_threshold:

            # Loop through height grid:
            IWV = 0.0
            k = 0
            prev_nonnan_idx = -1
            while k < n_height:

                # check if hum on current level is nan:
                # if so search for the next non-nan level:
                if np.isnan(q[k]):
                    next_nonnan_idx = np.where(~np.isnan(q[k:]))[0]

                    if (len(next_nonnan_idx) > 0) and (prev_nonnan_idx >= 0):   # mid or near top of height grid
                        next_nonnan_idx = next_nonnan_idx[0] + k    # plus k because searched over part of rho_v
                        IWV -= 0.25*(q[next_nonnan_idx] + q[prev_nonnan_idx])*(press[k+1] - press[k-1])
                    
                    elif (len(next_nonnan_idx) > 0) and (prev_nonnan_idx < 0):  # bottom of height grid
                        next_nonnan_idx = next_nonnan_idx[0] + k    # plus k because searched over part of q

                        # fixing height grid variable in case only the lowest measurement doesn't exist:
                        if np.isnan(press[0]) and not (np.isnan(press[1]+press[2])):
                            IWV -= 0.5*q[next_nonnan_idx]*(press[2] - press[1])
                        else:
                            IWV -= 0.5*q[next_nonnan_idx]*(press[k+1] - press[k])

                    else: # reached top of grid
                        IWV += 0.0

                else:
                    prev_nonnan_idx = k

                    if k == 0:          # bottom of grid
                        IWV -= 0.5*q[k]*(press[k+1] - press[k])
                    elif k == 1 and np.isnan(press[k-1]):       # next to bottom of grid
                        IWV -= 0.5*q[k]*(press[k+1] - press[k])
                    elif (k > 0) and (k < n_height-1):  # mid of grid
                        IWV -= 0.5*q[k]*(press[k+1] - press[k-1])
                    else:               # top of grid
                        IWV -= 0.5*q[k]*(press[-1] - press[-2])

                k += 1

        else:
            IWV = np.nan


    elif scheme == 'top_weighted':
        if (n_nans == 0):

            IWV = 0.0
            for k in range(n_height):
                if k < n_height-2:      # bottom or mid of grid
                    dp = press[k+1] - press[k]

                else:   # top and next to top of grid
                    dp = 0.5*(press[-1] - press[-2])        # half the height for top two levels

                IWV = IWV - q[k]*dp

        elif n_nans / n_height < nan_threshold:

            # Loop through height grid:
            IWV = 0.0
            k = 0
            prev_nonnan_idx = -1
            while k < n_height:
                
                # check if hum on current level is nan:
                # if so search for the next non-nan level:
                if np.isnan(q[k]):
                    next_nonnan_idx = np.where(~np.isnan(q[k:]))[0]

                    if (len(next_nonnan_idx) > 0) and (prev_nonnan_idx >= 0):   # mid of height grid
                        next_nonnan_idx = next_nonnan_idx[0] + k    # plus k because searched over part of q

                        if k+1 != n_height-1:
                            IWV -= 0.5*(q[next_nonnan_idx] + q[prev_nonnan_idx])*(press[k+1] - press[k])
                        else:   # near top of grid
                            IWV -= 0.25*(q[next_nonnan_idx] + q[prev_nonnan_idx])*(press[k+1] - press[k])
                    
                    elif (len(next_nonnan_idx) > 0) and (prev_nonnan_idx < 0):  # bottom of height grid
                        next_nonnan_idx = next_nonnan_idx[0] + k    # plus k because searched over part of q

                        # fixing height grid variable in case only the lowest measurement doesn't exist:
                        if np.isnan(press[0]) and not (np.isnan(press[1]+press[2])):
                            IWV -= q[next_nonnan_idx]*(press[2] - press[1])
                        else:
                            IWV -= q[next_nonnan_idx]*(press[k+1] - press[k])
                        

                    else: # reached top of grid
                        IWV += 0.0

                else:
                    prev_nonnan_idx = k

                    if k < n_height-2:  # bottom or mid of grid
                        IWV -= q[k]*(press[k+1] - press[k])
                    else:               # top of grid
                        IWV -= 0.5*q[k]*(press[-1] - press[-2])

                k += 1

        else:
            IWV = np.nan


    IWV = IWV / g       # yet had to be divided by gravitational acceleration

    return IWV


def e_sat(
    temp,
    which_algo='hyland_and_wexler'):

    """
    Calculates the saturation pressure over water after Goff and Gratch (1946)
    or Hyland and Wexler (1983).
    Source: Smithsonian Tables 1984, after Goff and Gratch 1946
    http://cires.colorado.edu/~voemel/vp.html
    http://hurri.kean.edu/~yoh/calculations/satvap/satvap.html

    e_sat_gg_water in Pa.

    Parameters:
    -----------
    temp : array of floats
        Array of temperature (in K).
    which_algo : str
        Specify which algorithm is chosen to compute e_sat (in Pa). Options:
        'hyland_and_wexler' (default), 'goff_and_gratch'
    """

    if which_algo == 'hyland_and_wexler':
        e_sat_gg_water = temp**(0.65459673e+01) * np.exp(-0.58002206e+04 / temp + 0.13914993e+01 - 0.48640239e-01*temp + 
                                0.41764768e-04*(temp**2) - 0.14452093e-07*(temp**3))

    elif which_algo == 'goff_and_gratch':
        e_sat_gg_water = 100 * 1013.246 * 10**(-7.90298*(373.16/temp-1) + 5.02808*np.log10(
                373.16/temp) - 1.3816e-7*(10**(11.344*(1-temp/373.16))-1) + 8.1328e-3 * (10**(-3.49149*(373.16/temp-1))-1))

    return e_sat_gg_water


def convert_rh_to_abshum(
    temp,
    relhum):

    """
    Convert array of relative humidity (between 0 and 1) to absolute humidity
    in kg m^-3. 

    Saturation water vapour pressure computation is based on: see e_sat(temp).

    Parameters:
    -----------
    temp : array of floats
        Array of temperature (in K).
    relhum : array of floats
        Array of relative humidity (between 0 and 1).
    """

    e_sat_water = e_sat(temp)

    rho_v = relhum * e_sat_water / (R_v * temp)

    return rho_v


def convert_rh_to_spechum(
    temp,
    pres,
    relhum):

    """
    Convert array of relative humidity (between 0 and 1) to specific humidity
    in kg kg^-1.

    Saturation water vapour pressure computation is based on: see e_sat(temp).

    Parameters:
    -----------
    temp : array of floats
        Array of temperature (in K).
    pres : array of floats
        Array of pressure (in Pa).
    relhum : array of floats
        Array of relative humidity (between 0 and 1).
    """

    e_sat_water = e_sat(temp)

    e = e_sat_water * relhum
    q = M_dv * e / (e*(M_dv - 1) + pres)

    return q
    
    
def convert_abshum_to_spechum(
    temp,
    pres,
    abshum):

    """
    Convert array of absolute humidity (kg m^-3) to specific humidity
    in kg kg^-1.

    Parameters:
    -----------
    temp : array of floats
        Array of temperature (in K).
    pres : array of floats
        Array of pressure (in Pa).
    abshum : array of floats
        Array of absolute humidity (in kg m^-3).
    """

    q = abshum / (abshum*(1 - 1/M_dv) + (pres/(R_d*temp)))

    return q


def q_to_h2ovmr(q: np.ndarray):
    
    """
    Converts specific humidity q (in kg kg-1) to volume mixing ratio (unitless).
    
    Parameters:
    -----------
    q : np.ndarray or xr.DataArray
        Specific humidity in kg kg-1.
    """
    
    return m_mol_air*q / (mw_h2o*(1.0 - q))


def h2ovmr_to_q(h2ovmr: np.ndarray):
    
    """
    Converts water vapour volume mixing ratio (unitless) to specific humidity q (in kg kg-1).
    
    Parameters:
    -----------
    h2ovmr : np.ndarray or xr.DataArray
        Water vapour volume mixing ratio.
    """
    
    return h2ovmr / ((m_mol_air/mw_h2o) + h2ovmr)


def rho_air(
    pres,
    temp,
    abshum):

    """
    Compute the density of air (in kg m-3) with a certain moisture load.

    Parameters:
    -----------
    pres : array of floats
        Array of pressure (in Pa).
    temp : array of floats
        Array of temperature (in K).
    abshum : array of floats
        Array of absolute humidity (in kg m^-3).
    """

    rho = (pres - abshum*R_v*temp) / (R_d*temp) + abshum

    return rho


def convert_spechum_to_abshum(
    temp,
    pres,
    q):

    """
    Convert array of specific humidity (kg kg^-1) to absolute humidity
    in kg m^-3.

    Parameters:
    -----------
    temp : array of floats
        Array of temperature (in K).
    pres : array of floats
        Array of pressure (in Pa).
    q : array of floats
        Array of specific humidity (in kg kg^-1).
    """

    abshum = pres / (R_d*temp*(1/q + 1/M_dv - 1))

    return abshum


def compute_heating_rate(
    upward_flux: np.ndarray,
    downward_flux: np.ndarray,
    rho: np.ndarray,
    height_lev: np.ndarray,
    convert_to_K_day=False):
    
    """
    Compute heating rates according to Petty (2006) chapter 10.4.1, equation 10.54 from
    upward and downward radiation fluxes (shortwave and longwave possible). Note that
    the radiation fluxes must be given on height levels while air density must be provided 
    on a height layer grid (whose boundaries are the height levels) because the heating rates
    will be put onto the height layer grid.
    
    Parameters:
    -----------
    upward_flux : np.ndarray or xr.DataArray
        Upward shortwave or longwave radiation flux in W m-2.
    downward_flux : np.ndarray or xr.DataArray
        Downward shortwave or longwave radiation flux in W m-2.
    rho : np.ndarray or xr.DataArray
        Air density (dry air + absolute humidity) in kg m-3.
    height_lev : np.ndarray or xr.DataArray
        Height levels (boundaries of height layers) in m.
    convert_to_K_day : bool
        If True, heating rates will be given in K day-1. Otherwise, in K s-1.
    """
    
    F_net = upward_flux - downward_flux
    dF_net_dz = np.diff(F_net, axis=-1) / np.diff(height_lev, axis=-1)
    HR = - dF_net_dz / (c_pd * rho)                 # heating rate in K s-1
    if convert_to_K_day: HR *= 86400.               # heating rate in K day-1
    
    return HR

In [ ]:
# plot_tools.py

def change_colormap_len(
    cmap, 
    n_new,
    to_listed_cmap=True):

    """
    Changes the number of colours of a given colourmap (cmap) to the number given by n_new.

    Parameters:
    -----------
    cmap : matplotlib.colors element
        Colourmap used by matplotlib.
    n_new : int
        Number of colours the new colourmap should have ('length of colourmap').
    to_listed_cmap : bool
        If true, returns a mpl.colors.ListedColormap object. If False, returns an array of 
        RGB-alpha values.
    """

    len_cmap = cmap.shape[0]    # length of colourmap
    n_rgba = cmap.shape[1]      # rgb + alpha

    cmap_new = np.zeros((n_new, n_rgba))
    for m in range(n_rgba):
        cmap_new[:,m] = np.interp(np.linspace(0, 1, n_new), np.linspace(0, 1, len_cmap), cmap[:,m])

    if to_listed_cmap: cmap_new = mpl.colors.ListedColormap(cmap_new)

    return cmap_new


def get_cm_cmap(name: str, to_listed_cmap=False):
    
    cmap = cm.__dict__[name](range(len(cm.__dict__[name].colors)))
    if to_listed_cmap:
        return mpl.colors.ListedColormap(cmap)
    else:
        return cmap


def stack_cm_cmaps(bounds: np.ndarray, cmap_names: list, breakpoints=np.array([])):
    
    """
    Merge two or more cmaps into one by stacking them. Breakpoints indicating
    at which points to switch from one to the next colourmap can be provided.
    """
    
    cmap_tuple = tuple([get_cm_cmap(cmap_name) for cmap_name in cmap_names])
    n_breaks = len(breakpoints)
    if n_breaks > 0:
        bounds_broken = list()
        for k, breakpoint in enumerate(breakpoints):
            if k == 0:
                bounds_broken.append(bounds[bounds < breakpoint])
            else:
                bounds_broken.append(bounds[(bounds >= breakpoints[k-1]) & (bounds < breakpoint)])
            
            if k == n_breaks-1:
                bounds_broken.append(bounds[bounds >= breakpoint])
        cmap_tuple = tuple([change_colormap_len(cmap, len(br_bound), False) for 
                             cmap, br_bound in zip(cmap_tuple, bounds_broken)])
    
    cmap_merged = np.vstack(cmap_tuple)
    cmap_merged = change_colormap_len(cmap_merged, len(bounds))
    
    return cmap_merged


def get_colourmap_kwargs(data_lims: np.ndarray, ddata: float, cmap_data, discrete=False):

    if discrete:
        data_lvls = np.arange(data_lims[0], data_lims[1]+ddata, ddata)
        n_lvls = len(data_lvls)
        norm_data = mpl.colors.BoundaryNorm(data_lvls, n_lvls)
        cmap_data = change_colormap_len(cmap_data, n_lvls, to_listed_cmap=True)
        
        kwargs = {'norm': norm_data, 'cmap': cmap_data}
    else:
        kwargs = {'vmin': data_lims[0], 'vmax': data_lims[1], 
                  'cmap': mpl.colors.ListedColormap(cmap_data)}
        
    return kwargs


def create_colourbar(
    fig: mpl.figure.Figure, 
    axis: mpl.axes.Axes, 
    image, 
    cb_label: str, 
    xpad=0.0, 
    cbwidth=0.03, 
    cb_kwargs=dict()):
    
    x0, y0, width, height = axis.get_position().bounds
    cax = fig.add_axes([x0+width+xpad, y0, cbwidth, height])
    cb = fig.colorbar(mappable=image, cax=cax, orientation='vertical', **cb_kwargs)
    cb.set_label(cb_label)
    
    return fig, axis


In [ ]:
# readers

categorize_model_translate_dict = {'temperature': 'temp', 
                                   'pressure': 'pres',
                                   'uwind': 'u',
                                   'vwind': 'v',
                                   'q': 'q'}
microphysics_translate_dict = {'der': 're_liq',
                               'der_scaled': 're_liq_scaled',
                               'der_error': 're_liq_error',
                               'der_scaled_error': 're_liq_scaled_error',
                               'der_retrieval_status': 're_liq_retrieval_status',
                               'ier_retrieval_status': 're_ice_retrieval_status',
                               'ier_error': 're_ice_error',
                               'ier': 're_ice'}

def read_cloudnet_categorize_model_data(
    path_data="",
    date0=None,
    date1=None,
    daterange=None,
    file_pattern="__DATE_STRING___lindenberg_categorize.nc",
    add_iwv=False):
    
    def preprocess_model_data(ds: xr.Dataset):
        
        keep_vars = ['temperature', 'pressure', 'q', 'uwind', 'vwind']
        data_vars = [*ds.data_vars]
        remove_vars = [var for var in data_vars if var not in keep_vars]
        ds = ds.drop_vars(remove_vars)
        ds = ds.rename_vars(categorize_model_translate_dict)
        
        ds = ds.drop_dims(['time', 'height'])
        ds = ds.rename({'model_time': 'time', 'model_height': 'height'})
        
        return ds
    
    if path_data == '': path_data = path_cloudnet_data
    
    files = get_files_daterange_filepattern(path_data=path_data,
                                            date0=date0,
                                            date1=date1,
                                            daterange=daterange,
                                            default_daterange=np.array([np.datetime64('2025-10-01')]),
                                            file_pattern=file_pattern)
    ds = read_files(files, process_fct=preprocess_model_data)
    
    if add_iwv:
        n_time = len(ds.time)
        ds['iwv'] = xr.DataArray(np.full((n_time,), np.nan), dims=['time'],
                                 attrs={'units': "kg m-2"})
        for k in range(n_time):
            ds['iwv'][k] = compute_IWV_q(ds['q'].isel(time=k).values, ds['pres'].isel(time=k).values)
    
    return ds.load()


def read_cloudnet_microphysics_retrievals_data(
    path_data="",
    date0=None,
    date1=None,
    daterange=None,):
    
    """
    Reads droplet and ice crystal effective radii, as well as liquid and ice water content data.
    """
    
    if path_data == '': path_data = path_cloudnet_data
    
    file_patterns = {'der': "__DATE_STRING___lindenberg_der.nc",
                     'ier': "__DATE_STRING___lindenberg_ier.nc",
                     'iwc': "__DATE_STRING___lindenberg_iwc-Z-T-method.nc",
                     'lwc': "__DATE_STRING___lindenberg_lwc-scaled-adiabatic.nc"}
    ds_dict = dict()
    for key, file_pattern in file_patterns.items():
        files = get_files_daterange_filepattern(path_data=path_data,
                                                date0=date0,
                                                date1=date1,
                                                daterange=daterange,
                                                default_daterange=np.array([np.datetime64('2025-10-01')]),
                                                file_pattern=file_pattern)
        
        ds_dict[key] = read_files(files)
    
    ds = xr.merge(ds_dict.values(), compat='no_conflicts')
    ds = add_height_levels_from_layers(ds)
    ds = add_iwc_lwc_for_levels(ds)
    
    ds = ds.rename_vars(microphysics_translate_dict)
    
    return ds.load()


def read_files(files: list, concat_dim='time', process_fct=None):
    
    ds = xr.open_mfdataset(files, concat_dim=concat_dim, combine='nested', 
                           preprocess=process_fct)
    ds = ds.sortby(concat_dim)
    
    return ds


def add_height_levels_from_layers(ds: xr.Dataset):
    
    height_lay = ds.height.values
    height_lev = 0.5*(height_lay[1:] + height_lay[:-1])
    height_lev_bot = np.array([2*height_lay[0] - height_lev[0]])
    height_lev_top = np.array([height_lay[-1] + 0.5*(height_lay[-1] - height_lay[-2])])
    height_lev = np.concatenate((height_lev_bot, 
                                 height_lev,
                                 height_lev_top))
    ds = ds.assign_coords({'height_lev': (['height_lev'], height_lev)})
    
    return ds


def add_iwc_lwc_for_levels(DS: xr.Dataset):
    
    n_time, n_hgt_lev = len(DS.time), len(DS.height_lev)
    for wc, wc_lev in zip(['iwc', 'lwc'], ['iwc_lev', 'lwc_lev']):
        DS[wc_lev] = xr.DataArray(np.zeros((n_time, n_hgt_lev), dtype=np.float64), dims=['time','height_lev'])
        DS[wc_lev][:,1:] = DS[wc].values
        
    return DS


hatpro_translate_dict = {
    'iwv': 'iwv',
    'lwp': 'lwp',
    'temperature': 'temp',
    'absolute_humidity': 'rho_v',
    'relative_humidity': 'relhum',
    'potential_temperature': 'pot_temp',
    'equivalent_potential_temperature': 'theta_e',
    }

def read_hatpro_derived_data(
    path_data="",
    vars_in=['iwv','lwp','temperature','absolute_humidity'],
    date0=None,
    date1=None,
    daterange=None,
    file_pattern="__DATE_STRING___lindenberg_hatpro-single_442ec2ea.nc",
    remove_bad_quality=False,
    apply_running_mean_sec=0):
    
    if path_data == '': path_data = path_hatpro_data
    
    files = get_files_daterange_filepattern(path_data=path_data,
                                            date0=date0,
                                            date1=date1,
                                            daterange=daterange,
                                            default_daterange=np.array([np.datetime64('2025-10-01')]),
                                            file_pattern=file_pattern)
    
    ds = xr.open_mfdataset(files, combine='nested', concat_dim='time')
    ds = ds.sortby('time')
    
    del_vars = [var for var in ds.data_vars if ((var not in vars_in) and (np.all(np.char.find(var, vars_in) == -1)))]
    ds = ds.drop_vars(del_vars)
    
    ds = rename_data_vars_incl_substrings(ds, hatpro_translate_dict)
    
    if remove_bad_quality:
        ds = set_bad_quality_to_nan(ds, good_quality_condition=lambda x: x == 0)
    
    if apply_running_mean_sec > 0:
        vars_for_running_mean = [hatpro_translate_dict[var_in] for var_in in vars_in]
        for var in vars_for_running_mean:
            if ds[var].ndim > 1: raise NotImplementedError
            ds[var][...] = running_mean_pdtime(ds[var].values, 
                                               apply_running_mean_sec, 
                                               ds[var].time.values)
    
    return ds.load()


def read_bsrn(
    path_data="",
    date0=None,
    date1=None,
    daterange=None,
    file_pattern="sups_rao_rad03_l1_any_v00___DATE_STRING__.nc"):
    
    if path_data == '': path_data = path_radiation_obs
    
    files = get_files_daterange_filepattern(path_data=path_data,
                                            date0=date0,
                                            date1=date1,
                                            daterange=daterange,
                                            default_daterange=np.array([np.datetime64('2025-10-01')]),
                                            file_pattern=file_pattern)
    
    ds = xr.open_mfdataset(files, combine='nested', concat_dim='time')
    ds = ds.rename({'DATETIME': 'time'})
    ds = ds.assign_coords({'time': ds.time})
    ds = ds.sortby('time')
    
    return ds


def read_pyrrtmg_simulation(
    path_data="",
    date="2025-10-01",
    hour=np.arange(24),
    file_pattern="lindenberg_radiation_sim___DATE_STRING__T__HOUR__.nc"):
    
    """
    Reads shortwave and longwave broadband radiation simulated with pyRRTMG for a given date
    and selected hours.
    
    Parameters:
    -----------
    path_data : str
        Full path where simulated radiation data is located. Can be empty, in which case the 
        default path is assumed.
    date : str
        Date in YYYY-MM-DD.
    hour : int or np.ndarray of int
        Hour(s) of the simulation.
    """
    
    if path_data == '': path_data = path_radiation_sim
    
    files = identify_hourly_files(path_data, date, hour, file_pattern=file_pattern)
    
    ds = xr.open_mfdataset(files, combine='nested', concat_dim='time')
    ds = ds.sortby('time')
    
    return ds.load()


def identify_hourly_files(path_data: str, date: str, hour, file_pattern: str):
    
    if isinstance(hour, int):
        file_pattern = file_pattern.replace("__HOUR__", f"{hour:02}")
    elif isinstance(hour, np.ndarray):
        file_pattern = file_pattern.replace("__HOUR__", "*")
    files = get_files_daterange_filepattern(path_data,
                                            date0=date,
                                            file_pattern=file_pattern)
    
    if isinstance(hour, np.ndarray):
        str_hour_array = np.char.add("T", np.char.zfill(hour.astype('str'), 2))
        file_pattern = file_pattern.replace("__DATE_STRING__", "*")
        
        files = np.array(files)
        files = [file for file in files if np.any(np.char.find(file, str_hour_array) != -1)]
    
    return files


meteo_var_translate_dict = {
    'air_pressure': 'pres',
    'air_temperature': 'temp',
    'relative_humidity': 'relhum',
    'wind_direction': 'wdir',
    'wind_speed': 'wspeed',
    'wind_speed_gust': 'wspeed_gust',
    'dew_point_temperature': 'dew_point',
    
    }

def read_weather_station_data(
    path_data="",
    date0=None,
    date1=None,
    daterange=None,
    file_pattern="__DATE_STRING___lindenberg_weather-station_ffb25f43.nc",
    remove_bad_quality=False):
    
    if path_data == '': path_data = path_meteo_obs
    
    files = get_files_daterange_filepattern(path_data=path_data,
                                            date0=date0,
                                            date1=date1,
                                            daterange=daterange,
                                            default_daterange=np.array([np.datetime64('2025-10-01')]),
                                            file_pattern=file_pattern)
    
    ds = xr.open_mfdataset(files, combine='nested', concat_dim='time')
    ds = ds.sortby('time')
    
    ds = rename_data_vars_incl_substrings(ds, meteo_var_translate_dict)
    
    if remove_bad_quality:
        ds = set_bad_quality_to_nan(ds, good_quality_condition=lambda x: x >= 4)
    
    return ds.load()


def rename_data_vars_incl_substrings(ds: xr.Dataset, translate_dict: dict):
    
    old_vars = np.asarray([*translate_dict.keys()])
    for dv in ds.data_vars:
        old_varname_in_dv = (np.char.find(dv, old_vars) != -1)
        if np.any(old_varname_in_dv):
            new_var_idx = np.where(old_varname_in_dv)[0][0]
            ds = ds.rename({dv: dv.replace(old_vars[new_var_idx], translate_dict[old_vars[new_var_idx]])})
    
    return ds


def set_bad_quality_to_nan(ds: xr.Dataset, quality_flag_name='quality_flag', good_quality_condition=lambda x: x == 0):
    
    data_vars = ds.data_vars
    for var in data_vars:
        var_qf = "_".join([var, "quality_flag"])
        if var_qf in data_vars:
            ds[var] = ds[var].where(good_quality_condition(ds[var_qf]), other=np.nan)

    return ds
 

def import_anderson_std_atm(file: str):
    
    """
    Imports greenhouse gas concentrations from standard atmospheres given by
    Anderson, G P, Clough, S A, Kneizys, F X, Chetwynd, J H, and Shettle, E P. AFGL (Air Force 
    Geophysical Laboratory) atmospheric constituent profiles (0. 120km). Environmental research 
    papers. United States: N. p., 1986. Web.
    Converts to volumne mixing ratios.
    
    Parameters:
    -----------
    file : str
        Full path to additional T-CARS data.
    """
    
    df = pd.read_table(file, sep='\s+',
                       names=('height','PPP','TTT','air','h2o','co2','o3','n2o','co','ch4','o2'),
                       dtype={'height': np.float32,'PPP': np.float32,'TTT': np.float32,'air': np.float32,
                              'h2o': np.float32,'co2': np.float32,'o3': np.float32,'n2o': np.float32,
                              'co': np.float32,'ch4':np.float32,'o2':np.float32}, skiprows=1)
    df.index = df['height']

    ds = xr.Dataset(coords={'height': (['height'], df.index.values*1000.)})
    for var in df.columns:
        if var in ds.coords: 
            continue
        elif var in ['h2o', 'co2', 'o3', 'n2o', 'co', 'ch4', 'o2']:
            df[var] = sfac['ppm'] * df[var].values
        ds[var] = xr.DataArray(df[var].values, dims=['height'])
    
    return ds


def import_std_atm(file: str):
    
    """
    Imports height, pressure, temperature, water vapour, CO2, O3, N2O, CO, CH4 and O2 data from
    University of Oxford, National Centre for Earth Observation - Natural Environmental Research 
    Council: RFM Atmospheric Profiles - FASCODE/ICRCCM Model Atmospheres, 
    https://eodg.atm.ox.ac.uk/RFM/atm/
    This data also seems to be the same as in
    Anderson, G P, Clough, S A, Kneizys, F X, Chetwynd, J H, and Shettle, E P. AFGL (Air Force 
    Geophysical Laboratory) atmospheric constituent profiles (0. 120km). Environmental research 
    papers. United States: N. p., 1986. Web.
    This reader is also based on the read_atm.py provided in https://eodg.atm.ox.ac.uk/RFM/atm/.
    Converts gas data from ppmv to volume mixing ratios, and uses SI units elsewhere.
    
    Parameters:
    -----------
    file : str
        Full path and file name to standard atmosphere data.
    """
    
    vmr_data = ['h2o', 'co2', 'o3', 'n2o', 'co', 'ch4', 'o2']
    rename_dict = {'pre': 'pres',
                   'tem': 'temp',}
    unit_conv_dict = {'pres': [0., 100.]}
    for vmr_ in vmr_data:
        rename_dict[vmr_] = vmr_
        unit_conv_dict[vmr_] = [0., sfac['ppm']]
    
    with open(file) as f:
        rec = '!'
        while rec[0] == '!': rec = f.readline()  # skip initial comments
        flds = rec.split()
        nlev = int(flds[0])
        atm = { 'nlev':nlev } 
        rec = f.readline()
        while rec[0:4].lower() != '*end':  # repeat until end marker record
            if rec[0] == '!': continue 
            flds = rec.split()
            key = flds[0][1:].lower() # remove '*' and change to lower case
            prf = np.fromfile(f,sep=", ",count=nlev)
            atm[key] = prf
            rec = f.readline()
            if rec == '': break   # also exit if end-of-file without *END
    
    ds = xr.Dataset(coords={'height': (['height'], atm['hgt']*1000.)})
    for key, val in rename_dict.items():
        ds[val] = xr.DataArray(atm[key], dims=['height'])
    
    for var in unit_conv_dict:
        ds[var] = convert_units(ds[var], unit_conv_dict[var])
    
    return ds


def import_trace_gas_csv(file):

    """
    Imports the NOAA Trace Gas data from "https://gml.noaa.gov/aggi/aggi.html" and puts it
    into an xarray data set.

    Parameters:
    -----------
    file : str
        Full path and filename of trace gas concentration data in a .csv table.
    """
    
    def get_species_units(lines: np.ndarray):
        
        varnames = lines[2]
        tg_species = [species.lower().replace('-', '') for species in varnames if species != ""]
        units = [unit for unit in lines[3] if unit != ""]
        assert len(tg_species) == len(units)
        
        return tg_species, units
    
    def to_vmr_units(DS: xr.Dataset, tg_species: list, units: list):
        
        for unit, var in zip(units, tg_species):
            DS[var] *= sfac[unit]
        
        return DS
    
    with open(file, newline='') as csvfile:
        csv_reader = csv.reader(csvfile, delimiter=',')
        list_of_lines = [row for row in csv_reader]
    
    lines = np.array(list_of_lines)
    n_lines = lines.shape[0]
    first_footer_line_no = np.where(np.all(lines == '', axis=1))[0][0]
    
    tg_species, units = get_species_units(lines)

    header = 3
    DF = pd.read_csv(file, sep=',', header=header, index_col=0, 
                     names=tg_species, na_values='nd',
                     nrows=n_lines - (n_lines-first_footer_line_no)-header-1)
    DF.index.name = 'time'
    DS = DF.to_xarray()
    
    time = np.floor(DS.time.values).astype('int64')
    time = np.asarray([np.datetime64(f"{t}-07-01T00:00:00") for t in time])
    DS = DS.assign_coords({'time': (['time'], time.astype('datetime64[ns]'))})
    
    DS = to_vmr_units(DS, tg_species, units)

    return DS

In [ ]:
# tcars.py

class tcars:
    
    """
    Loads all necessary constants for radiative transfer simulations using T-CARS (python interface of RRTMG)*.
    
    *: Deneke, H. (2024) hdeneke/pyRRTMG: Release with correct versioning scheme . Zenodo [code]. https://doi.org/10.5281/zenodo.11147087
    
    Init:
    path_tcars_data : str
        Full path where additional data for T-CARS is stored.
    ds : xr.dataset
        Dataset containing atmospheric data. Must have "time" and "height" dimensions.
    """
    
    def __init__(self, path_tcars_data: str, ds: xr.Dataset):
        
        self.ds = ds
        self.path_tcars_data = path_tcars_data
        
        # for more detailed descriptions of the flags below, see 
        # https://github.com/hdeneke/pyRRTMG/blob/master/src_f/rrtmg_sw/column_model/doc/rrtmg_sw_instructions.txt
        self.icld = 1       # icld = 1 if clouds should be included; icld = 0 if clouds are omitted; 
                            # flag values 2 or 3 change the "overlap assumption"
        self.iaer = 0       # iaer = 10: one or more layers contain aerosols; = 0 to omit aerosols
        self.permuteseed_sw = 0     # permuteseed_sw
        self.permuteseed_lw = 0     # permuteseed_lw
        self.irng = 0               # irng
        self.idrv = 0               # idrv https://github.com/hdeneke/pyRRTMG/tree/master/src_f/rrtmg_lw -> 4th paragraph
                                    # --> whether to also calculate the derivative of flux with respect to surface temp
        
        self.adjes = 1.0        # adjustment of total solar irradiance (solar constant)
        self.solar_constant = solar_constant      # solar constant; total solar irradiance at TOA
        self.inflgsw = 2            # 0: direct specif. of cloud opt depth, cloud fraction, single scat albedo, phase function
                                    # 2: ice and liq cloud opt depths are calculated from lwp, iwp, effective radii
        self.inflglw = 2            # similar to inflgsw; see also INFLAG in pyRRTMG doc noted above
        self.iceflgsw = 2           # see ICEFLAG in pyRRTMG doc
        self.iceflglw = 2
        self.liqflgsw = 1           # see LIQFLAG in pyRRTMG doc
        self.liqflglw = 1
        
        # flags for the use of gas volume mixing ratios: 
        # -1: turn off, _vmr is set to 0
        # 0: use reference given in standard atmosphere (Anderson 1986)
        # 1: use values given in DS
        self.iflag_h2o_vmr = 1
        self.iflag_co2_vmr = 0      # additional option: 2: use NOAA measurements
        self.iflag_o3_vmr = 0
        self.iflag_n2o_vmr = 0      # additional option: 2: use NOAA measurements
        self.iflag_ch4_vmr = 0      # additional option: 2: use NOAA measurements
        self.iflag_o2_vmr = 0
        
        self.iflag_emissivity = 0   # -1: use emissivity of 1, 0: use emissivity of 0.996, 
                                    # 1: user-defined emissivity in set_emissivity
                
        self.set_translation_dict()
        self.init_cloud_properties()
        self.init_aerosol_properties()
        self.set_emissivity()
        
        
    def load_background_vmrs(self, which_std_atm='subarctic_summer'):
        
        """
        Loads trace gas data from modeled standard atmosphere and observations.
        """
        
        def std_atm_data_to_tcars_time_height_grid(ds: xr.Dataset):
            
            ds = ds.expand_dims(dim={'time': self.ds.time.values}, axis=0)
            ds = ds.interp(coords={'height': self.ds.height.values}, kwargs={"fill_value": 'extrapolate'})
            
            return ds
        
        def trace_gas_obs_to_tcars_time_height_grid(ds: xr.Dataset):
            
            if ((ds.time.values[-1] < self.ds.time.values[0]) or 
                (ds.time.values[0] > self.ds.time.values[-1])):
                ds = ds.sel(time=self.ds.time.values, method='nearest')
                ds = ds.assign_coords({'time': (['time'], self.ds.time.values)})
            else:
                ds = ds.interp(time=self.ds.time.values).expand_dims(dim={'height': self.ds.height.values}, axis=1)
            ds = ds.expand_dims(dim={'height': self.ds.height.values}, axis=1)
            
            return ds
        
        std_atm_filename_dict = {
            'subarctic_summer': "sas.atm",
            'subarctic_winter': "saw.atm",
            'midlat_summer': "mls.atm",
            'midlat_winter': "mlw.atm",
            'std': "std.atm",
            'tropical': "tro.atm",
            }
        
        filename = os.path.join(self.path_tcars_data, 
                                "std_atm/",
                                std_atm_filename_dict[which_std_atm])
        ds = import_std_atm(filename)
        ds = std_atm_data_to_tcars_time_height_grid(ds)
        
        n_time, n_hgt = len(self.ds.time), len(self.ds.height)
        dims_list = ['time', 'height']
        for var in ['h2o', 'co2', 'o3', 'n2o', 'co', 'ch4', 'o2']:
            var_vmr = var + "_vmr"
            
            if ((f'iflag_{var_vmr}' in self.__dict__.keys()) and (self.__dict__[f'iflag_{var_vmr}'] == 0) or 
                (f'iflag_{var_vmr}' not in self.__dict__.keys())):
                self.ds[var_vmr] = xr.DataArray(ds[var].values, dims=dims_list)
        

        add_ghgs = {'cfc11': 232.0, 
                    'cfc12': 516.0, 
                    'cfc22': 233.0, 
                    'ccl4': 82.0}       # all in ppt, Source: NOAA
        for var in add_ghgs:
            self.ds[var+"_vmr"] = xr.DataArray(np.full((n_time, n_hgt), add_ghgs[var]*sfac['ppt']), 
                                               dims=dims_list)
        
        ds = import_trace_gas_csv(self.path_tcars_data + "/trace_gas_data/NOAA_Annual_Mean_MoleFractions.csv")
        ds = trace_gas_obs_to_tcars_time_height_grid(ds)

        for var in ['co2', 'n2o', 'ch4', 'cfc11', 'cfc12', 'ccl4']:
            var_vmr = var + "_vmr"
            if (((f'iflag_{var_vmr}' in self.__dict__.keys()) and (self.__dict__[f'iflag_{var_vmr}'] == 2)) or 
                (f'iflag_{var_vmr}' not in self.__dict__.keys())):
                self.ds[var_vmr] = xr.DataArray(ds[var].values, dims=dims_list)

    
    def set_rrtmg_input(self, which_std_atm: str):
        
        """
        Forward data loaded into Dataset to dictionaries used by T-CARS (=RRTMG).
        
        Parameters:
        -----------
        which_std_atm : str
            Defines which standard atmosphere to load. The following options are supported:
            - "subarctic_summer"
            - "subarctic_winter"
            - "midlat_summer"
            - "midlat_winter"
            - "std"
            - "tropical"
        """

        self.load_background_vmrs(which_std_atm=which_std_atm)
        self.update_DS()
        self.update_cloud_properties()
        self.update_aerosol_properties()

        atm_args = ['Play', 'Plev',                                  # pressure of layer and level in hPa
                    'Tlay', 'Tlev', 'Tsfc',                          # temperature of layer and level (layer boundaries) in K
                    'h2ovmr', 'o3vmr', 'co2vmr', 'ch4vmr', 'n2ovmr', # vol mix ratio of gases
                    'o2vmr', 'cfc11vmr', 'cfc12vmr', 'ccl4vmr', 'cfc22vmr']
        
        atm = dict()
        for ds_var, t_var in self.trl_.items():
            if t_var in atm_args:
                try:
                    atm[t_var] = np.asfortranarray(self.ds[ds_var], dtype=np.float64)
                except KeyError:
                    print(f"{ds_var} is needed but was not found in the dataset provided to {os.path.basename(__file__)}" + 
                          f" while executing {sys.argv[0]}.")
                
                
        for var in [
                    'alb_dir_uv',           # UV/VIS surface albedo, direct (0.2-0.625 um)
                    'alb_dif_uv',           # UV/VIS surface albedo, diffuse (0.2-0.625 um)
                    'alb_dir_nir',          # near-IR surface albedo, direct (0.625-12.20 um)
                    'alb_dif_nir',          # near-IR surface albedo, diffuse (0.625-12.20 um)
                    'julian_day',           # "Day of the year" or Julian Day number
                    'cos_zenith',           # cosine of the solar zenith angle
                    ]:
            
            if var in self.ds.data_vars:
                self.__setattr__(var, self.ds[var].values)
            
            # Fill values if data not available in DS:
            elif var == 'julian_day':
                self.julian_day = 1
            elif var in ["alb_dir_uv", "alb_dif_uv", "alb_dir_nir", "alb_dif_nir"]:
                self.__setattr__(var, 0.1)
            elif var == 'cos_zenith':
                self.cos_zenith = 0.0
        
        
        self.rrtmg_input = [self.icld,
                            self.iaer,
                            self.permuteseed_sw,
                            self.permuteseed_lw,
                            self.irng,
                            self.idrv]
        
        self.rrtmg_input += [atm[k] for k in atm_args]
        
        self.rrtmg_input += [self.alb_dif_nir, 
                             self.alb_dir_nir, 
                             self.alb_dif_uv, 
                             self.alb_dir_uv,
                             self.emis,
                             self.cos_zenith,
                             self.adjes,
                             self.julian_day,
                             self.solar_constant,
                             self.inflgsw,
                             self.inflglw,
                             self.iceflgsw,
                             self.iceflglw,
                             self.liqflgsw,
                             self.liqflglw]
        
        for var in self.cloud_props.keys():
            self.rrtmg_input += [self.cloud_props[var]]
        for var in self.aerosol_props.keys():
            self.rrtmg_input += [self.aerosol_props[var]]
        
        self.rrtmg_input = odict(zip(rrtmg.input_vars, self.rrtmg_input))
        
        
        for var in ['h2o', 'co2', 'o3', 'n2o', 'ch4', 'o2']:
            if self.__dict__[f'iflag_{var}_vmr'] == -1:
                self.rrtmg_input[var+"vmr"][...] = 0
    
    
    def set_translation_dict(self):
        
        """
        Translation dict to rename variables of self.ds (keys) to the names used by T-CARS 
        (values).
        """
        
        self.trl_ = {'pres': 'Play',
                     'temp': 'Tlay',
                     'temp_sfc': 'Tsfc',
                     'pres_h': 'Plev',
                     'temp_h': 'Tlev',
                     'clwp': 'lwp',
                     'ciwp': 'iwp',
                     'clc': 'cldfrac',
                     're_liq': 're_liq',
                     're_ice': 're_ice',
                     'tauc_sw': 'tauc_sw',
                     'tauc_lw': 'tauc_lw',
                     'ssac_sw': 'ssac_sw',
                     'asmc_sw': 'asmc_sw',
                     'fsfc_sw': 'fsfc_sw',
                     }
        
        for var in ['h2o_vmr', 'co2_vmr', 'o3_vmr', 'n2o_vmr', 'ch4_vmr', 'o2_vmr',
                    'cfc11_vmr', 'cfc12_vmr', 'cfc22_vmr', 'ccl4_vmr']:
            self.trl_[var] = var.replace('_vmr', 'vmr')
    
    
    def init_cloud_properties(self):
        
        self.cloud_props = dict()
        for var in [
                    'tauc_sw',      # in cloud optical depth, short wave
                    'tauc_lw',      # in cloud optical depth, long wave
                    'cldfrac',      # cloud fraction
                    'ssac_sw',      # in cloud single scattering albedo
                    'asmc_sw',      # in cloud assymetry parameter
                    'fsfc_sw',      # in cloud forward scattering fraction, 
                                    # delta function pointing forward; forward peaked scattering
                    'iwp',          # cloud ice water path, in g m-2
                    'lwp',          # cloud liquid water path, in g m-2
                    're_ice',       # effective radius ice, in um
                    're_liq',       # effective radius liquid, in um
                    ]:
            
            shape_ = (len(self.ds.time), len(self.ds.height))
            fill_val = 0.0
            if "_sw" in var:
                shape_ = (rrtmg.nbnd_sw,) + shape_
            elif "_lw" in var:
                shape_ = (rrtmg.nbnd_lw,) + shape_
                
            self.cloud_props[var] = np.full(shape_, fill_val, dtype=np.float64, order='F')
    
    
    def init_aerosol_properties(self):
        
        shape_base = (len(self.ds.time), len(self.ds.height))
        
        kw = {'dtype': np.float64, 'order': "F"}
        self.aerosol_props = {
            "tauaer_sw": np.full(shape_base + (rrtmg.nbnd_sw,), 0.0, **kw),    
                         # aerosol opt depth (iaer=10 only); (ncol,nlay,nbndsw)
                         # iaer=10 means one or more layers contain aerosols
            "ssaaer_sw": np.full(shape_base + (rrtmg.nbnd_sw,), 1.0, **kw),    
                         # aerosol single scat albedo (iaer=10 only)
            "asmaer_sw": np.full(shape_base + (rrtmg.nbnd_sw,), 0.0, **kw),
                         # aerosol assymetry parameter (iaer=10 only)
            "ecaer_sw": np.full(shape_base + (6,), 0.0, **kw),
                        # aerosol opt depth at 0.55 um (iaer=6 only)
            "tauaer_lw": np.full(shape_base + (rrtmg.nbnd_lw,), 0.0, **kw),
                         # aerosol opt depth at mid-point of LW spectral bands
            }
    
    
    def update_DS(self):
        
        """
        Update temperature data on height layers and use temperature of lowest level as surface 
        temperature. This ensures consistency when editing temperature levels.
        """
        
        self.ds['temp'][...] = 0.5*(self.ds['temp_h'].values[...,:-1] + self.ds['temp_h'].values[...,1:])
    
    
    def update_cloud_properties(self):
        
        self.cloud_props = self.update_properties(self.cloud_props)
            
            
    def update_aerosol_properties(self):
        
        self.aerosol_props = self.update_properties(self.aerosol_props)
    
    
    def update_properties(self, props: dict):
        
        prop_args = [*props.keys()]
        for var in self.ds.data_vars:
            
            if var in prop_args:
                var_update = var
            elif (var in self.trl_.keys()) and (self.trl_[var] in prop_args):
                var_update = self.trl_[var]
            else:
                continue
            props[var_update] = self.ds[var].values
        
        return props
    
    
    def set_emissivity(self, custom_emissivity=0.9837):
        
        """
        Emissivity: 16 bands in longwave, see 
        https://github.com/hdeneke/pyRRTMG/blob/master/src_f/rrtmg_lw/column_model/doc/rrtmg_lw_instructions.txt
        -> 3-1000 um
        
        Use 0.999 for sea ice fraction >= 0.5.
        """
        
        array_kwargs = dict(dtype=np.float64, order='F')
        emis_shape = (len(self.ds.time), 16)
        if self.iflag_emissivity == -1:
            emis_val = 1.0
        elif self.iflag_emissivity == 0:
            emis_val = 0.996            # as in Thielke et al. 2022, https://doi.org/10.1038/s41597-022-01461-9
        elif self.iflag_emissivity == 1:
            # define your own emissivity array of emis_shape or use the following:
            emis_val = custom_emissivity
        self.emis = np.ones(emis_shape, **array_kwargs) * emis_val
    
    
    def run_tcars(self):
        
        self.run_sanity_test()
        self.flxhr = rrtmg.calc_flxhr(**self.rrtmg_input)
        self.organise_output()
        
        
    def run_sanity_test(self):
        
        for var in ['play', 'plev', 'tlay', 'tlev', 'clwp', 'ciwp', 'h2ovmr',
                    'o3vmr', 'co2vmr', 'ch4vmr', 'n2ovmr', 'o2vmr', 'cfc11vmr',
                    'cfc12vmr', 'cfc22vmr', 'ccl4vmr']:
            assert np.all(self.rrtmg_input[var] >= 0.), f"{var} is insane! Consider debugging..."
        
        relq_msg = ("Effective radius of liq. droplets seems out of valid bounds [2.5, 60] um " +
                    "(or 0 um for cloudfree height layers).")
        assert np.all((self.rrtmg_input['relq'] >= 0.) & (self.rrtmg_input['relq'] <= 60.)), relq_msg
        
        reic_msg = ("Effective radius of ice particles seems out of valid bounds [13, 130] um " +
                    "(or 0 um for cloudfree height layers).")
        assert np.all((self.rrtmg_input['reic'] >= 0.) & (self.rrtmg_input['reic'] <= 130.)), reic_msg
    
    
    def organise_output(self):
        
        def correct_uppermost_shortwave_heating_rate():
            
            self.out_ds['swhr'][{'height': -1}] = self.out_ds['swhr_raw'].isel(height=-1)
        
        self.out_ds = xr.Dataset(coords=self.ds.coords)
        
        n_time, n_hgt, n_hgt_h = len(self.ds.time), len(self.ds.height), len(self.ds.height_h)
        shape_lay = (n_time, n_hgt)
        shape_lev = (n_time, n_hgt_h)
        for var, flx in self.flxhr.items():
            
            dims_list = ['time']
            if flx.shape == shape_lay:
                dims_list.append('height')
            elif flx.shape == shape_lev:
                dims_list.append('height_h')
            self.out_ds[var] = xr.DataArray(flx, dims=dims_list)
        
        
        # pyRRTMG seems to return incorrect heating rates for the uppermost height layer. 
        # Compute it also manually:
        spec_hum = h2ovmr_to_q(self.ds.h2o_vmr)
        rho_v = convert_spechum_to_abshum(self.ds.temp, self.ds.pres*100., spec_hum)
        rho = rho_air(self.ds.pres*100., self.ds.temp, rho_v)
        for sky_cond in ['', 'c']:
            for band in ['sw', 'lw']:
                HR_ = compute_heating_rate(self.out_ds[f'{band}uflx{sky_cond}'], 
                                           self.out_ds[f'{band}dflx{sky_cond}'], 
                                           rho, self.ds.height_h, convert_to_K_day=True)
                self.out_ds[f'{band}hr{sky_cond}_raw'] = HR_
        
        correct_uppermost_shortwave_heating_rate()
        
        
        # information about the variable names and their meanings: 
        # https://github.com/hdeneke/pyRRTMG/blob/master/src_f/_rrtmg.f90 , paragraph "! Output"
        self.out_ds['height'].attrs = {'long_name': "Height of the full levels (centre of layer)", 
                                       'units': "m"}
        self.out_ds['height_h'].attrs = {'long_name': "Height of the half levels (layer boundaries)", 
                                         'units': "m"}
        self.out_ds['swuflx'].attrs = {'long_name': "Upward shortwave flux at half level",
                                       'units': "W m-2"}
        self.out_ds['swdflx'].attrs = {'long_name': "Downward shortwave flux at half level",
                                       'units': "W m-2"}
        self.out_ds['swdirflx'].attrs = {'long_name': "Direct shortwave flux at half level",
                                       'units': "W m-2"}
        self.out_ds['swhr'].attrs = {'long_name': "Shortwave radiative heating rates for layers",
                                       'units': "K day-1"}
        self.out_ds['swuflxc'].attrs = {'long_name': "Clear sky upward shortwave flux at half level",
                                       'units': "W m-2"}
        self.out_ds['swdflxc'].attrs = {'long_name': "Clear sky downward shortwave flux at half level",
                                       'units': "W m-2"}
        self.out_ds['swdirflxc'].attrs = {'long_name': "Clear sky direct shortwave flux at half level",
                                       'units': "W m-2"}
        self.out_ds['swhrc'].attrs = {'long_name': "Clear sky shortwave radiative heating rates for layers",
                                       'units': "K day-1"}
        self.out_ds['lwuflx'].attrs = {'long_name': "Upward longwave flux at half level",
                                       'units': "W m-2"}
        self.out_ds['lwdflx'].attrs = {'long_name': "Downward longwave flux at half level",
                                       'units': "W m-2"}
        self.out_ds['lwhr'].attrs = {'long_name': "Longwave radiative heating rates for layers",
                                       'units': "K day-1"}
        self.out_ds['lwuflxc'].attrs = {'long_name': "Clear sky upward longwave flux at half level",
                                       'units': "W m-2"}
        self.out_ds['lwdflxc'].attrs = {'long_name': "Clear sky downward longwave flux at half level",
                                       'units': "W m-2"}
        self.out_ds['lwhrc'].attrs = {'long_name': "Clear sky longwave radiative heating rates for layers",
                                       'units': "K day-1"}
        
        self.out_ds.attrs['title'] = "RRTMG outputs from T-CARS environment"
    
    
    def save_output(self, path: str, filename: str):
        
        os.makedirs(path, exist_ok=True)
        
        self.out_ds = write_basic_attributes(self.out_ds)
        self.out_ds = update_netCDF_file_history(DS=self.out_ds,
                                                 script_name=os.path.basename(__file__),
                                                 summary_str="create simulations",
                                                 history_attr_exists=False)
        
        self.out_ds = encode_time(self.out_ds)
        
        outfile = os.path.join(path, filename)
        self.out_ds.to_netcdf(outfile, mode='w', format='NETCDF4')
        print(f"Saved {outfile}....")

In [ ]:
# radiation_simulations.py


def prepare_tcars_client_for_sim(tcars_client):
    
    tcars_client = customise_gas_flags(tcars_client)
    tcars_client = set_emissivity(tcars_client, 0.98) # based on Jin and Liang, 2006, https://doi.org/10.1175/JCLI3720.1
    
    return tcars_client


def customise_gas_flags(tcars_client):
    
    tcars_client.iflag_co2_vmr = 2
    tcars_client.iflag_o3_vmr = 0
    tcars_client.iflag_n2o_vmr = 2
    tcars_client.iflag_ch4_vmr = 2
    tcars_client.iflag_o2_vmr = 0
    
    return tcars_client


def set_emissivity(tcars_client, emissivity: float):
    
    tcars_client.iflag_emissivity = 1
    tcars_client.set_emissivity(custom_emissivity=emissivity)
    
    return tcars_client


def prepare_tcars_ds(
    timeline: np.ndarray,
    meteo_ds: xr.Dataset, 
    mp_ds: xr.Dataset,
    height_grid: np.ndarray,
    weather_station_ds=None,
    hatpro_ds=None):
    
    """
    For certain timeline over which the simulation is supposed to run, extract pressure, 
    temperature and humidity data from meteo data set (meteo_ds) and extract microphysical
    data (droplet effective radii, ice crystal effective radii, ice and liquid water contents) 
    from mp_ds. The data will be put onto levels ('half levels' or 'layer boundaries') and 
    layers ('full levels').
    """
    
    meteo_ds = prepare_meteo_dataset(meteo_ds, timeline, height_lev=height_grid)
    meteo_ds = modify_meteo_data(meteo_ds, weather_station_ds, hatpro_ds)
    mp_ds = prepare_micophysics_dataset(mp_ds, timeline, height_lev=height_grid)
    
    ds = init_tcars_ds(time=timeline,
                       height_h=height_grid,
                       height=0.5*(height_grid[:-1] + height_grid[1:]))
    
    ds = meteo_to_tcars_ds(ds, meteo_ds)
    ds = add_solar_cos_zen_lindenberg(ds)
    ds = set_albedo(ds)
    ds = add_cloud_data(ds, mp_ds)
    
    return ds


def add_cloud_data(ds: xr.Dataset, mp_ds: xr.Dataset):
        
    def cloud_sanity_enforcer(ds: xr.Dataset):
        
        bounds = {'re_liq': np.array([2.5, 60.]),
                  're_ice': np.array([13., 130.])}
        corresponding_water_path = {'re_liq': 'clwp',
                                    're_ice': 'ciwp'}
        for k, val in bounds.items():
            wp = corresponding_water_path[k]
            re_between_0_and_lower_bound = ((ds[k] > 0.) & (ds[k] < val[0]))
            ds[k] = ds[k].where(~re_between_0_and_lower_bound, other=val[0])
            ds[wp] = ds[wp].where(~re_between_0_and_lower_bound, other=val[0])
            
            radius_or_water_path_is_zero = (ds[k] == 0.) | (ds[wp] == 0.)
            ds[k] = ds[k].where(~radius_or_water_path_is_zero, other=0.)
            ds[wp] = ds[wp].where(~radius_or_water_path_is_zero, other=0.)
        
        return ds
    
    cloud_vars = ['re_liq', 're_ice', 'ciwp', 'clwp']
    for var in cloud_vars:
        if var in ds.data_vars:
            ds[var][...] = mp_ds[var].values
    
    # if (ds['clwp'] > 0).any():
    #     ds['clwp'] = scale_microphysics_lwp_ret_by_hatpro_obs(ds['clwp'], hat_ds['lwp'],
    #                                                          fill_mask=cloud_vars.fill_mask)
    
    ds = cloud_sanity_enforcer(ds)
    
    ds['clc'] = ds['clc'].where((ds.re_liq == 0.) & 
                                (ds.re_ice == 0.) &
                                (ds.ciwp == 0.) &
                                (ds.clwp == 0.), other=1.)
    
    return ds


def prepare_micophysics_dataset(
    mp_ds: xr.Dataset, 
    timeline: np.ndarray, 
    height_lev: np.ndarray,
    height_lay=None):
    
    def convert_to_rrtmg_units(ds: xr.Dataset):
        
        """
        From kg m-2 to g m-2. And from m to um.
        """
        
        for var in ['clwp', 'ciwp']: ds[var] *= 1000.
        for var in ['re_liq', 're_ice']: ds[var] *= 1e+06
        
        return ds
    
    if height_lay is None: height_lay = 0.5*(height_lev[:-1] + height_lev[1:])
    
    mp_ds = mp_ds.interp({'time': timeline}, method='linear')
    mp_ds = mp_ds.interp({'height': height_lay,
                          'height_lev': height_lev},
                         method='linear',
                         kwargs={'fill_value': 0.})
    for var in ['re_liq', 're_liq_scaled', 're_ice', 'iwc', 'lwc', 'iwc_lev', 'lwc_lev']:
        mp_ds[var] = mp_ds[var].where(~mp_ds[var].isnull(), other=0.)
    mp_ds = get_layer_water_paths(mp_ds)
    
    mp_ds = convert_to_rrtmg_units(mp_ds)
    
    return mp_ds


def get_layer_water_paths(ds: xr.Dataset):
    
    """
    Compute layer-integrated cloud LWP and IWP (not the total vertical integral of LWC and IWC!).
    ciwp and clwp are height-resolved variables, where e.g.
    ciwp[k] = iwc[k] * (height_lev[k+1] - height_lev[k])
    """
    
    for wc, wp in zip(['iwc', 'lwc'], ['ciwp', 'clwp']):
        wp_values = ds[wc].values*ds.height_lev.diff('height_lev').values
        ds[wp] = xr.DataArray(wp_values, dims=['time', 'height'])
    
    return ds


def set_albedo(DS: xr.Dataset):
    
    """
    Set surface albedo for Lindenberg research site.
    """
    
    for var in ['alb_dir_uv', 'alb_dif_uv']:
        DS[var][...] = 0.2
    for var in ['alb_dir_nir', 'alb_dif_nir']:
        DS[var][...] = 0.2
    
    return DS


def add_solar_cos_zen_lindenberg(ds: xr.Dataset):
    
    lat, lon = 52.208, 14.118
    sun_ele = get_sun_elevation_time_loc(time=ds.time.values, lat=lat, lon=lon)
    cos_zen = np.cos(np.deg2rad(90. - sun_ele))
    ds['cos_zenith'] = xr.DataArray(cos_zen, dims=['time'])
    
    return ds


def get_sun_elevation_time_loc(time: np.ndarray, lat: float, lon: float):
    
    time_sky = load_skyfield_time(time)
    load = custom_skyfield_loader()
    
    sun, earth = create_observable_object(loader=load)
    loc = earth + api.wgs84.latlon(lat, lon, elevation_m=11)
    sun_ele = compute_sun_ele_for_loc(loc, time_sky, sun)
    
    return sun_ele


def load_skyfield_time(time: np.ndarray):
    
    ts = api.load.timescale()
    time_xr = xr.DataArray(time.astype('datetime64[ns]'), dims=['time'],
                           coords={'time': (['time'], time.astype('datetime64[ns]'))})
    time_sky = ts.utc(time_xr.dt.year.values, time_xr.dt.month.values, time_xr.dt.day.values,
                      time_xr.dt.hour.values, time_xr.dt.minute.values)
    
    return time_sky


def custom_skyfield_loader():
    
    return Loader(os.path.join(os.environ['PATH_DATA_BASE'], "skyfield/"))


def create_observable_object(loader=None):
    
    if loader is None:
        eph = api.load("de421.bsp")
    else:
        eph = loader("de421.bsp")
    
    sun = eph['sun']
    earth = eph['earth']
    
    return sun, earth


def compute_sun_ele_for_loc(loc, time, sun):
    
    """
    Observe sun from given location (loc) for a given time. Get azimuth and altitude (elevation)
    angle.
    """
    
    sun_astro = loc.at(time).observe(sun)
    sun_ele_azi = sun_astro.apparent().altaz()
    sun_ele = sun_ele_azi[0].degrees
    
    return sun_ele


def meteo_to_tcars_ds(ds: xr.Dataset, meteo_ds: xr.Dataset):
    
    def meteo_quality_control(ds: xr.Dataset):
        
        assert ((ds.temp > 170.) & (ds.temp < 330.)).all()
        assert ((ds.pres < 1200.)).all()
        assert ((ds.h2o_vmr >= 0.)).all()
        assert (~(ds.temp + ds.pres + ds.h2o_vmr).isnull()).all()
        assert (~(ds.temp_h + ds.pres_h).isnull()).all()
        
        return ds
    
    for var in ['pres', 'temp', 'h2o_vmr']:
        ds[var+"_h"] = xr.DataArray(meteo_ds[var].values,
                                    dims=['time', 'height_h'])
        ds[var] = xr.DataArray(0.5*(meteo_ds[var].isel(height=slice(1,None)).values +
                                    meteo_ds[var].isel(height=slice(None,-1)).values),
                               dims=['time', 'height'])
    
    ds['temp_sfc'][...] = meteo_ds['temp'].isel(height=0).values
    
    ds = ds.drop_vars('h2o_vmr_h')
    ds = meteo_quality_control(ds)
    
    return ds


def modify_meteo_data(meteo_ds: xr.Dataset, weather_station_ds=None, hatpro_ds=None):
    
    def replace_near_sfc_air_temp(meteo_ds: xr.Dataset, temp_2m: xr.DataArray):
        
        temp_2m = temp_2m.interp({'time': meteo_ds.time})
        temp_2m = temp_2m.ffill('time').bfill('time')
        meteo_ds['temp'][{'height': 0}] = temp_2m.values
        
        return meteo_ds
    
    def scale_spec_humidity_profile_with_hatpro_iwv(meteo_ds: xr.Dataset, iwv: xr.DataArray):
        
        n_time = len(meteo_ds.time)
        meteo_ds['iwv'] = xr.DataArray(np.full((n_time,), np.nan), dims=['time'],
                                 attrs={'units': "kg m-2"})
        pres_si = meteo_ds['pres']*100.
        for k in range(n_time):
            meteo_ds['iwv'][k] = compute_IWV_q(meteo_ds['q'].isel(time=k).values, 
                                               pres_si.isel(time=k).values)
        
        iwv = iwv.interp({'time': meteo_ds.time})
        iwv = iwv.ffill('time').bfill('time')
        meteo_ds['q'] *= iwv/meteo_ds['iwv']
        
        return meteo_ds
    
    if weather_station_ds is not None:
        meteo_ds = replace_near_sfc_air_temp(meteo_ds, weather_station_ds.temp)
    if hatpro_ds is not None:
        meteo_ds = scale_spec_humidity_profile_with_hatpro_iwv(meteo_ds, hatpro_ds.iwv)
        meteo_ds['h2o_vmr'] = q_to_h2ovmr(meteo_ds.q)
    
    return meteo_ds


def prepare_meteo_dataset(meteo_ds: xr.Dataset, timeline: np.ndarray, height_lev: np.ndarray):
    
    meteo_ds['h2o_vmr'] = q_to_h2ovmr(meteo_ds.q)
    meteo_ds['pres'] *= 0.01        # Pa to hPa
    
    meteo_ds = meteo_ds.interp({'time': timeline}, method='linear')
    meteo_ds = meteo_ds.interp(height=height_lev)
    
    return meteo_ds


def init_tcars_ds(
    time: np.ndarray, 
    height_h: np.ndarray,
    height=None):
    
    if height is None:
        height = 0.5*(height_h[...,1:] + height_h[...,:-1])
    
    DS = xr.Dataset(coords={'time': (['time'], time),
                            'height_h': (['height_h'], height_h),
                            'height': (['height'], height)})
    
    n_time = len(time)
    n_hgt_h = len(height_h)
    n_hgt = len(height)
    
    cloud_vars = ['clwp',        # in-cloud liquid water path in g m-2
                  'ciwp',        # in-cloud ice water path in g m-2
                  'clc',        # cloud cover
                  're_liq',     # effective radius liq in um
                  're_ice',     # effective radius ice in um
                  ]
    atmos_vars = ['pres_h',     # pressure at half levels in hPa
                  'pres',       # pressure at full levels in hPa
                  'temp_h',     # temperature at half levels in K
                  'temp',       # temperature at full levels in K
                  'h2o_vmr',    # water vapour volume mixing ratio at full levels
                  ]
    gas_vars = ['co2_vmr',
                'o3_vmr',
                'n2o_vmr',
                'ch4_vmr',
                'o2_vmr']
    sfc_vars = ['alb_dir_uv',   # direct albedo UV spectrum 
                'alb_dif_uv',   # diffuse albedo UV spectrum
                'alb_dir_nir',  # direct albedo near IR spectrum
                'alb_dif_nir',  # diffuse albedo near IR spectrum
                'temp_sfc',     # surface temperature in K
                ]     
    vars = cloud_vars + atmos_vars + gas_vars + sfc_vars
    
    for var in vars:
        if var in ['pres_h', 'temp_h']:
            shape = (n_time,n_hgt_h)
            dims = ('time', 'height_h')
        elif var in sfc_vars:
            shape = (n_time,)
            dims = ('time')
        else:
            shape = (n_time,n_hgt)
            dims = ('time', 'height')
        
        if var in (cloud_vars + gas_vars + sfc_vars):
            fill_val = 0.
        elif var in atmos_vars:
            fill_val = np.nan
        
        DS[var] = xr.DataArray(np.full(shape, fill_val), dims=dims)
    
    DS['julian_day'] = DS.time.dt.dayofyear
    
    return DS


def set_hourly_simulation_timeline(date: np.datetime64, hour: int, time: xr.DataArray):
    
    timeline = time.sel(time=date.astype('datetime64[D]').astype('str')).sel(time=time.dt.hour == hour).values
    
    return timeline


def define_height_grid(surface_height=0.):
    
    height_grid0 = np.arange(surface_height, 5000., 10)
    height_grid1 = np.arange(height_grid0[-1] + 10., 10000., 20)
    height_grid2 = np.arange(height_grid1[-1] + 20., 25000., 50)
    height_grid = np.concatenate((height_grid0, height_grid1, height_grid2))
    
    return height_grid


def data_overview_quicklooks(
    path_plots: str,
    cn_model_ds: xr.Dataset, 
    cn_mp_ds: xr.Dataset,
    save_figures=False,
    date_str="2025-10-01"):
    
    import matplotlib as mpl
    mpl.use("WebAgg")
    import matplotlib.pyplot as plt
    
    def plot_data(ds: xr.Dataset, 
                  vars: list, 
                  cmaps: list, 
                  vmins: list, 
                  vmaxs: list,
                  ylims=None,
                  filename=f"lindenberg_overview_{date_str}"):
        
        f1, axs = plt.subplots(4, 1, sharex=True, figsize=(12,10))
    
        for ax, var, cmap_str, vmin, vmax in zip(axs, 
                                                 vars, 
                                                 cmaps, 
                                                 vmins,
                                                 vmaxs):
            cmap = get_cm_cmap(cmap_str, to_listed_cmap=True)
            img = ax.pcolormesh(ds.time, ds.height, ds[var].T,
                                cmap=cmap, vmin=vmin, vmax=vmax)
            
            f1, ax = create_colourbar(f1, ax, img, cb_label=f"{var} ({ds[var].units})")
            ax.set_ylabel("Height (m)")
            ax.set_xlabel(", ".join(["Time", date_str]))
            if ylims is not None: ax.set_ylim(ylims)
            ax.label_outer()
        
        if save_figures:
            os.makedirs(path_plots, exist_ok=True)
            
            plotfile = os.path.join(path_plots, filename + ".png")
            f1.savefig(plotfile, dpi=200)

            print(f"Saved {plotfile} ....")
            
        else:
            plt.show()
            pdb.set_trace()
            
        plt.close()

    date_str_short = date_str.replace("-", "")
    
    vars_model = ['temp', 'q', 'u', 'v']
    cmaps_model = ['batlow', 'oslo_r', 'vik', 'vik']
    vmins_model = [200, 0.0, -40, -40]
    vmaxs_model = [290, 0.006, 40, 40]
    vars_mp = ['re_liq', 're_ice', 'iwc', 'lwc']
    cmaps_mp = ['batlow', 'batlow', 'batlow', 'batlow']
    vmins_mp = [None, None, None, None]
    vmaxs_mp = [None, None, None, None]
    vars_mp_err = ['re_liq_error', 're_ice_error', 'iwc_error', 'lwc_error']
    cmaps_mp_err = ['batlow', 'batlow', 'batlow', 'batlow']
    vmins_mp_err = [None, None, None, None]
    vmaxs_mp_err = [None, None, None, None]
    vars_mp_ret = ['re_liq_retrieval_status', 're_ice_retrieval_status', 'iwc_retrieval_status', 'lwc_retrieval_status']
    cmaps_mp_ret = ['batlow', 'batlow', 'batlow', 'batlow']
    vmins_mp_ret = [None, None, None, None]
    vmaxs_mp_ret = [None, None, None, None]
    plot_data(cn_model_ds, vars_model, cmaps_model, vmins_model, vmaxs_model,
              ylims=[0., 25000.],
              filename=f"lindenberg_cloudnet_model_overview_{date_str_short}")
    plot_data(cn_mp_ds, vars_mp, cmaps_mp, vmins_mp, vmaxs_mp,
              ylims=[0,4000],
              filename=f"lindenberg_cloudnet_microphysics_overview_{date_str_short}")
    plot_data(cn_mp_ds, vars_mp_err, cmaps_mp_err, vmins_mp_err, vmaxs_mp_err,
              ylims=[0,4000],
              filename=f"lindenberg_cloudnet_microphysics_err_overview_{date_str_short}")
    plot_data(cn_mp_ds, vars_mp_ret, cmaps_mp_ret, vmins_mp_ret, vmaxs_mp_ret,
              ylims=[0,4000],
              filename=f"lindenberg_cloudnet_microphysics_ret_overview_{date_str_short}")


In [ ]:
# Site and case study specific constants

date_str = "2025-10-01"                     # only "2025-10-01" is available
date = np.datetime64(date_str)
station_ele_amsl = 104.                      # station elevation above mean sea level in m (manually extracted from cloudnet data)

With all these functions defined, we can now set up the radiation simulations. You may modify the paths, height grid and other settings here. Note, however, if you would like to change the path where to save the radiation simulations, please modify `path_radiation_sim` in the **Paths** cell at the beginning of this notebook.

### Settings

In [ ]:
# Settings:

path_output = path_radiation_sim
path_plots = os.environ['PATH_PLOTS_BASE']              # uses a default setting

height_grid = define_height_grid(station_ele_amsl)      # always make sure that the height grid starts with station_ele_amsl
replace_2m_temp_by_obs = True
scale_hum_profile_by_hatpro_iwv = True

data_quicklooks = False
set_dict = {'save_figures': False,          # whether or not to save plots to file
            'date_str': date_str}

For the simulations, we need to load the model and microphysics data from the Cloudnet retrieval. Optionally, we can also load weather station and HATPRO observations with which the model profile could be manipulated.

### Loading data

In [ ]:
# Loading data

weather_station_ds = None
hatpro_ds = None
if replace_2m_temp_by_obs:
    weather_station_ds = read_weather_station_data(date0=date, remove_bad_quality=True)
if scale_hum_profile_by_hatpro_iwv:
    hatpro_ds = read_hatpro_derived_data(vars_in=['iwv'], 
                                            date0=date, 
                                            remove_bad_quality=True,
                                            apply_running_mean_sec=60)
cn_model_ds = read_cloudnet_categorize_model_data(date0=date)
cn_mp_ds = read_cloudnet_microphysics_retrievals_data(date0=date)
if data_quicklooks: data_overview_quicklooks(path_plots, cn_model_ds, cn_mp_ds, **set_dict)

Here, you may add further modification steps to the input data.

### Data modifications

In [ ]:
# empty for now... you could, for example, add a few Kelvin to the model temperature profile
# cn_model_ds['temp'] += 2.

The loaded data can now be prepared for the simulations (interpolate to the time and height grid of the simulations) and forwarded to the data set that gathers all needed data for the simulations.

### Prepare and run simulations

In [ ]:
for hour in range(24):
    timeline = set_hourly_simulation_timeline(date=date, hour=hour, time=cn_mp_ds.time)
    
    ds = prepare_tcars_ds(timeline, 
                            deepcopy(cn_model_ds), 
                            deepcopy(cn_mp_ds), 
                            height_grid, 
                            deepcopy(weather_station_ds),
                            deepcopy(hatpro_ds))
    tcars_client = tcars(path_tcars_data=path_tcars_data, ds=ds)
    tcars_client = prepare_tcars_client_for_sim(tcars_client)
    
    tcars_client.set_rrtmg_input(which_std_atm="midlat_summer")
    tcars_client.run_tcars()
    
    filename_tcars_sim = f"lindenberg_radiation_sim_{date_str.replace('-','')}T{hour:02}.nc"
    tcars_client.save_output(path_output, filename_tcars_sim)